# Test direct response traces

This notebook runs both a built-in Operator and an Agent Studio workflow against the local Narada development stack. It displays the typed `response.trace` records as a tree and as raw JSON, then validates their IDs, timestamps, and parent relationships.

Before starting Jupyter, run the local backend, the frontend from the response-trace PR, and the local extension build. Load the SDK API key from `.env`, then launch the notebook from the SDK repository:

```sh
set -a
source .env
set +a
uv run --with jupyter jupyter lab
```

The notebook defaults to `http://localhost:8000/fast/v2` for the backend and `http://localhost:3000/initialize` for the frontend.

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import socket
import tempfile
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from IPython.display import JSON, display
from narada import Agent, AgentKind, BrowserConfig, BrowserEnvironment, Span, Trace

## Configure the local stack

The notebook launches Chrome for Testing with the development extension. `NARADA_API_KEY_DEV` is accepted as a convenience alias for `NARADA_API_KEY`.

In [ ]:
DEV_EXTENSION_ID = "ijdopnjleolkjakldkjplfhniiohnccf"
WORKSPACE_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "narada-python-sdk").is_dir() and (path / "caddie").is_dir()
)
EXTENSION_PATH = (
    WORKSPACE_ROOT
    / "caddie"
    / "src"
    / "google"
    / "chrome-extension"
    / ".output"
    / "chrome-mv3-dev"
)

if dev_api_key := os.getenv("NARADA_API_KEY_DEV"):
    os.environ["NARADA_API_KEY"] = dev_api_key
if "NARADA_API_KEY" not in os.environ:
    raise RuntimeError(
        "Set NARADA_API_KEY_DEV or NARADA_API_KEY before starting Jupyter."
    )
if not (EXTENSION_PATH / "manifest.json").is_file():
    raise RuntimeError("Build the development extension with `npm run local` first.")

os.environ.setdefault("NARADA_API_BASE_URL", "http://localhost:8000/fast/v2")
INITIALIZATION_URL = os.getenv(
    "NARADA_INITIALIZATION_URL",
    "http://localhost:3000/initialize",
)

print("Workspace:", WORKSPACE_ROOT)
print("Extension:", EXTENSION_PATH)
print("API:", os.environ["NARADA_API_BASE_URL"])

## Start an isolated development browser

This uses a temporary Chrome profile and attaches the SDK to the same Chrome process that loaded the local extension.

In [ ]:
def find_chrome_for_testing() -> Path:
    cache_roots = (
        Path.home() / "Library" / "Caches" / "ms-playwright",
        Path.home() / ".cache" / "ms-playwright",
    )
    patterns = (
        "chromium-*/chrome-mac*/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing",
        "chromium-*/chrome-mac*/Chromium.app/Contents/MacOS/Chromium",
        "chromium-*/chrome-linux*/chrome",
        "chromium-*/chrome-win*/chrome.exe",
    )
    candidates = [
        executable
        for cache_root in cache_roots
        for pattern in patterns
        for executable in cache_root.glob(pattern)
        if executable.is_file() and os.access(executable, os.X_OK)
    ]
    if not candidates:
        raise RuntimeError(
            "Chrome for Testing is missing. Run `uv run playwright install chromium`."
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


def find_free_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def get_loaded_extension_ids(cdp_port: int) -> set[str]:
    with urllib.request.urlopen(
        f"http://127.0.0.1:{cdp_port}/json/list",
        timeout=1,
    ) as response:
        targets = json.load(response)

    prefix = "chrome-extension://"
    return {
        url.removeprefix(prefix).partition("/")[0]
        for target in targets
        if isinstance(target, dict)
        and isinstance((url := target.get("url")), str)
        and url.startswith(prefix)
    }


async def wait_for_dev_extension(
    browser_process: asyncio.subprocess.Process,
    cdp_port: int,
) -> None:
    seen_extension_ids: set[str] = set()
    for _ in range(100):
        if browser_process.returncode is not None:
            raise RuntimeError(
                f"Chrome exited during startup with code {browser_process.returncode}."
            )
        try:
            seen_extension_ids = await asyncio.to_thread(
                get_loaded_extension_ids, cdp_port
            )
        except OSError:
            pass
        if DEV_EXTENSION_ID in seen_extension_ids:
            return
        await asyncio.sleep(0.1)

    seen = ", ".join(sorted(seen_extension_ids)) or "none"
    raise RuntimeError(
        f"The development extension did not load. Extension IDs seen: {seen}."
    )

In [ ]:
browser_user_data = tempfile.TemporaryDirectory(prefix="narada-response-trace-")
browser_config = BrowserConfig(
    executable_path=str(find_chrome_for_testing()),
    user_data_dir=browser_user_data.name,
    cdp_host="http://127.0.0.1",
    cdp_port=find_free_port(),
    initialization_url=INITIALIZATION_URL,
    extension_id=DEV_EXTENSION_ID,
)
browser_process = await asyncio.create_subprocess_exec(
    browser_config.executable_path,
    f"--user-data-dir={browser_config.user_data_dir}",
    f"--profile-directory={browser_config.profile_directory}",
    "--remote-debugging-address=127.0.0.1",
    f"--remote-debugging-port={browser_config.cdp_port}",
    f"--disable-extensions-except={EXTENSION_PATH}",
    f"--load-extension={EXTENSION_PATH}",
    "--no-default-browser-check",
    "--no-first-run",
    "about:blank",
    stdout=asyncio.subprocess.DEVNULL,
    stderr=asyncio.subprocess.DEVNULL,
)

try:
    await wait_for_dev_extension(browser_process, browser_config.cdp_port)
    environment = BrowserEnvironment(config=browser_config, attach_to_existing=True)
    await environment.start()
except BaseException:
    if browser_process.returncode is None:
        browser_process.terminate()
        await browser_process.wait()
    browser_user_data.cleanup()
    raise

print("Browser window ID:", environment.browser_window_id)

## Trace display and validation helpers

The API returns a flat list. The tree is reconstructed only from each span's `parent_id`; it does not use `action_trace` or `workflow_trace`.

In [ ]:
def exported_models(records: list[Any] | None) -> list[dict[str, Any]]:
    return [record.model_dump(mode="json", by_alias=True) for record in records or []]


def span_type(span: Span[Any]) -> str:
    return span.span_data.type


def span_tree_label(span: Span[Any]) -> str:
    data = span.span_data
    detail = (
        getattr(data, "workflow_name", None)
        or getattr(data, "name", None)
        or getattr(data, "message", None)
    )
    status = getattr(data, "status", None)
    suffix = ""
    if detail:
        suffix += f": {detail}"
    if status:
        suffix += f" [{status}]"
    if span.error is not None:
        suffix += f" ERROR={span.error.message}"
    return f"{data.type}{suffix}"


def validate_trace(records: list[Trace | Span[Any]]) -> dict[str, Any]:
    traces = [record for record in records if isinstance(record, Trace)]
    spans = [record for record in records if isinstance(record, Span)]
    assert len(traces) == 1, f"Expected one Trace record, found {len(traces)}"
    trace = traces[0]
    span_ids = {span.span_id for span in spans}
    assert len(span_ids) == len(spans), "Span IDs must be unique"

    for span in spans:
        assert span.trace_id == trace.trace_id, (
            "Every span must use the trace header ID"
        )
        assert span.started_at is not None, f"{span.span_id} is missing started_at"
        assert span.ended_at is not None, f"{span.span_id} is missing ended_at"
        if span.parent_id is not None:
            assert span.parent_id in span_ids, f"{span.span_id} has a missing parent"

    return {
        "trace_id": trace.trace_id,
        "name": trace.name,
        "span_count": len(spans),
        "span_types": dict(Counter(span_type(span) for span in spans)),
    }


def show_trace_tree(records: list[Trace | Span[Any]]) -> None:
    trace = next(record for record in records if isinstance(record, Trace))
    spans = [record for record in records if isinstance(record, Span)]
    children: dict[str | None, list[Span[Any]]] = defaultdict(list)
    for span in spans:
        children[span.parent_id].append(span)

    print(f"Trace: {trace.name} ({trace.trace_id})")

    def print_children(parent_id: str | None, prefix: str) -> None:
        child_spans = children[parent_id]
        for index, span in enumerate(child_spans):
            is_last = index == len(child_spans) - 1
            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{span_tree_label(span)}")
            child_prefix = prefix + ("    " if is_last else "│   ")
            print_children(span.span_id, child_prefix)

    print_children(None, "")


def inspect_response(response: Any) -> None:
    summary = validate_trace(response.trace)
    display(summary)
    show_trace_tree(response.trace)
    print("\nRaw response.trace:")
    display(JSON(exported_models(response.trace), expanded=False))
    print(
        "\nIndependent legacy fields:",
        {
            "action_trace_items": len(response.action_trace or []),
            "has_workflow_trace": response.workflow_trace is not None,
        },
    )

## Run Operator

This should produce a Generalist agent span, a nested Operator span, and direct `agent_action` spans for the browser work.

In [ ]:
operator = Agent(environment=environment, kind=AgentKind.OPERATOR)
operator_response = await operator.run(
    prompt="Open https://example.com and tell me the page heading and current URL.",
    timeout=300,
)

print("Response:", operator_response.text)
inspect_response(operator_response)

operator_span_types = {
    span_type(record) for record in operator_response.trace if isinstance(record, Span)
}
assert "agent" in operator_span_types
assert "agent_action" in operator_span_types

## Run an Agent Studio workflow

Set `WORKFLOW_PATH` to a GUI workflow accessible to the API key. The public greeter is a convenient smoke-test default, but use one of your own GUI workflows to guarantee `workflow`, `gui_step.*`, and control-flow spans. A useful test workflow contains Start → Set Variable → For Loop → Print → Output.

In [ ]:
WORKFLOW_PATH = os.getenv(
    "NARADA_DEMO_WORKFLOW",
    "/demo@narada.ai/greeter-agent",
)
WORKFLOW_PROMPT = os.getenv("NARADA_DEMO_WORKFLOW_PROMPT", "John Doe")

print("Workflow:", WORKFLOW_PATH)
print("Prompt:", WORKFLOW_PROMPT)

In [ ]:
workflow = Agent(environment=environment, kind=WORKFLOW_PATH)
workflow_response = await workflow.run(prompt=WORKFLOW_PROMPT, timeout=300)

print("Response:", workflow_response.text)
inspect_response(workflow_response)

workflow_span_types = {
    span_type(record) for record in workflow_response.trace if isinstance(record, Span)
}
if "workflow" not in workflow_span_types:
    print(
        "No workflow span was returned. Replace WORKFLOW_PATH with a GUI workflow "
        "owned by or shared with the API-key user."
    )
elif not any(span_type.startswith("gui_step.") for span_type in workflow_span_types):
    print("The workflow ran, but it did not execute an SDK-shaped GUI step.")

## Compare the two traces

In [ ]:
display(
    {
        "operator": validate_trace(operator_response.trace),
        "workflow": validate_trace(workflow_response.trace),
    }
)

## Cleanup

Run this cell even if one of the examples fails.

In [ ]:
await environment.close(timeout=30)
if browser_process.returncode is None:
    browser_process.terminate()
    try:
        await asyncio.wait_for(browser_process.wait(), timeout=10)
    except TimeoutError:
        browser_process.kill()
        await browser_process.wait()
browser_user_data.cleanup()
print("Closed the SDK environment and development browser.")